# Day 18: Regular Expressions (Regex)

**Learning objectives:** master regex patterns, clean text data, extract patterns, validate formats.

**Contents**
1. Regex basics
2. Python `re` module
3. Common patterns (email, phone, URL, date, IP)
4. Pattern library (20+ patterns, tested)
5. Data cleaning with regex
6. Practice: customer data + log parser

In [1]:
import re
import pandas as pd

## 1. Regex Basics

### 1.1 Literal characters
Plain text matches itself exactly.

In [2]:
text = "The cat sat on the mat."
print(re.findall(r"cat", text))      # exact word
print(re.findall(r"at", text))       # every 'at'

['cat']
['at', 'at', 'at']


### 1.2 Metacharacters
| Symbol | Meaning |
|---|---|
| `.` | any character (except newline) |
| `\d` / `\D` | digit / non-digit |
| `\w` / `\W` | word char (letter, digit, `_`) / non-word |
| `\s` / `\S` | whitespace / non-whitespace |
| `\.` | a literal dot (escape with backslash) |
| `\|` | OR |

In [3]:
text = "Order 66 shipped on 2024-01-15 for $9.99"
print("Any char   :", re.findall(r"c.t", "cat cot cut c-t"))
print("Digits     :", re.findall(r"\d+", text))
print("Words      :", re.findall(r"\w+", text))
print("Whitespace :", len(re.findall(r"\s", text)))
print("Literal dot:", re.findall(r"\d+\.\d+", text))
print("OR         :", re.findall(r"cat|dog", "I have a cat and a dog"))

Any char   : ['cat', 'cot', 'cut', 'c-t']
Digits     : ['66', '2024', '01', '15', '9', '99']
Words      : ['Order', '66', 'shipped', 'on', '2024', '01', '15', 'for', '9', '99']
Whitespace : 6
Literal dot: ['9.99']
OR         : ['cat', 'dog']


### 1.3 Character classes  `[...]`

In [4]:
print(re.findall(r"[aeiou]", "regular expressions"))       # vowels
print(re.findall(r"[A-Z]", "Hello World From Python"))     # capitals
print(re.findall(r"[0-9a-f]+", "id: 3fa9 and zz 12ab"))    # hex-like
print(re.findall(r"[^0-9]+", "abc123def456"))              # NOT digits

['e', 'u', 'a', 'e', 'e', 'i', 'o']
['H', 'W', 'F', 'P']
['d', '3fa9', 'a', 'd', '12ab']
['abc', 'def']


### 1.4 Quantifiers
| Symbol | Meaning |
|---|---|
| `*` | 0 or more |
| `+` | 1 or more |
| `?` | 0 or 1 |
| `{n}` | exactly n |
| `{n,m}` | between n and m |

In [5]:
print(re.findall(r"colou?r", "color colour"))       # ? optional
print(re.findall(r"\d{3}", "12 123 1234"))          # exactly 3
print(re.findall(r"\d{2,4}", "1 12 123 12345"))     # 2 to 4
print(re.findall(r"ab*", "a ab abbb"))              # * zero or more b
print(re.findall(r"<.+?>", "<b>bold</b>"))          # lazy match (+?)
print(re.findall(r"<.+>", "<b>bold</b>"))           # greedy match

['color', 'colour']
['123', '123']
['12', '123', '1234']
['a', 'ab', 'abbb']
['<b>', '</b>']
['<b>bold</b>']


### 1.5 Anchors
`^` start of string, `$` end of string, `\b` word boundary.

In [6]:
print(bool(re.search(r"^Hello", "Hello World")))    # starts with
print(bool(re.search(r"World$", "Hello World")))    # ends with
print(re.findall(r"\bcat\b", "cat concatenate cat"))  # whole word only

True
True
['cat', 'cat']


### 1.6 Groups
`( )` captures, `(?: )` doesn't capture, `(?P<name> )` names a group.

In [7]:
m = re.search(r"(\d{4})-(\d{2})-(\d{2})", "Date: 2024-03-15")
print(m.group(0), "|", m.groups())          # whole match | all groups

m = re.search(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})", "Date: 2024-03-15")
print(m.groupdict())                         # named groups

print(re.findall(r"(?:Mr|Ms|Dr)\. (\w+)", "Dr. Smith met Ms. Jones"))   # non-capturing prefix
print(re.sub(r"(\w+) (\w+)", r"\2 \1", "John Smith"))                  # swap with backrefs

2024-03-15 | ('2024', '03', '15')
{'year': '2024', 'month': '03', 'day': '15'}
['Smith', 'Jones']
Smith John


## 2. Python `re` Module

In [8]:
text = "Call 555-1234 or 555-9876. Office: 555-0000"

# re.search -> first match ANYWHERE in the string
print("search :", re.search(r"\d{3}-\d{4}", text).group())

# re.match -> match only at the START of the string
print("match  :", re.match(r"\d{3}-\d{4}", text))              # None (starts with 'Call')
print("match  :", re.match(r"Call", text).group())

# re.findall -> list of ALL matches
print("findall:", re.findall(r"\d{3}-\d{4}", text))

# re.sub -> replace
print("sub    :", re.sub(r"\d{3}-\d{4}", "XXX-XXXX", text))

# re.compile -> pre-build a pattern for reuse (faster, cleaner)
phone = re.compile(r"\d{3}-\d{4}")
print("compile:", phone.findall(text))

search : 555-1234
match  : None
match  : Call
findall: ['555-1234', '555-9876', '555-0000']
sub    : Call XXX-XXXX or XXX-XXXX. Office: XXX-XXXX
compile: ['555-1234', '555-9876', '555-0000']


### Flags

In [9]:
# re.IGNORECASE (re.I) -> ignore upper/lower case
print(re.findall(r"python", "Python PYTHON python", re.I))

# re.MULTILINE (re.M) -> ^ and $ match at each line
multi = "line one\nline two\nline three"
print(re.findall(r"^line", multi, re.M))

# re.DOTALL (re.S) -> dot also matches newline
print(re.findall(r"one.*two", multi, re.S))

# re.VERBOSE (re.X) -> allow comments and whitespace in the pattern
date_pat = re.compile(r"""
    (\d{4})   # year
    -(\d{2})  # month
    -(\d{2})  # day
""", re.X)
print(date_pat.search("2024-03-15").groups())

['Python', 'PYTHON', 'python']
['line', 'line', 'line']
['one\nline two']
('2024', '03', '15')


## 3. Common Patterns

In [10]:
# Email validation
email_re = re.compile(r"^[\w.+-]+@[A-Za-z0-9-]+(\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}$")
for e in ["john@example.com", "a.b+tag@mail.co.uk", "bad@", "no at.com", "x@y"]:
    print(f"{e:22} ->", bool(email_re.match(e)))

john@example.com       -> True
a.b+tag@mail.co.uk     -> True
bad@                   -> False
no at.com              -> False
x@y                    -> False


In [11]:
# Phone number extraction (US style)
phone_re = re.compile(r"\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}")
text = "Call (555) 123-4567, 555.987.6543 or 555 111 2222."
print(phone_re.findall(text))

['(555) 123-4567', '555.987.6543', '555 111 2222']


In [12]:
# URL matching
url_re = re.compile(r"https?://(?:www\.)?[\w-]+(?:\.[\w-]+)+(?:/[^\s]*)?")
text = "Visit https://www.python.org/docs or http://example.com/a?b=1 today"
print(url_re.findall(text))

['https://www.python.org/docs', 'http://example.com/a?b=1']


In [13]:
# Date parsing (YYYY-MM-DD and DD/MM/YYYY)
text = "Start 2024-03-15, end 20/04/2024"
print(re.findall(r"\d{4}-\d{2}-\d{2}", text))
print(re.findall(r"\d{2}/\d{2}/\d{4}", text))

['2024-03-15']
['20/04/2024']


In [14]:
# IP address validation (each part must be 0-255)
octet = r"(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)"
ip_re = re.compile(rf"^{octet}(?:\.{octet}){{3}}$")
for ip in ["192.168.1.1", "255.255.255.255", "256.1.1.1", "1.2.3", "10.0.0.01"]:
    print(f"{ip:18} ->", bool(ip_re.match(ip)))

192.168.1.1        -> True
255.255.255.255    -> True
256.1.1.1          -> False
1.2.3              -> False
10.0.0.01          -> False


## 4. Pattern Library (22 patterns)

A dictionary of ready-to-use patterns. Each has a **valid** and **invalid** example, and the cell
below tests all of them automatically.

In [15]:
PATTERNS = {
    # name: (pattern, valid example, invalid example)
    "email":          (r"^[\w.+-]+@[A-Za-z0-9-]+(\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}$", "me@site.com", "me@site"),
    "phone_us":       (r"^\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}$", "(555) 123-4567", "12-34"),
    "phone_intl":     (r"^\+\d{1,3}[\s-]?\d{6,12}$", "+91 9876543210", "9876543210"),
    "url":            (r"^https?://[\w-]+(\.[\w-]+)+(/\S*)?$", "https://python.org/doc", "python.org"),
    "ipv4":           (r"^(?:(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)\.){3}(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)$", "192.168.0.1", "300.1.1.1"),
    "mac_address":    (r"^(?:[0-9A-Fa-f]{2}:){5}[0-9A-Fa-f]{2}$", "00:1A:2B:3C:4D:5E", "00:1A:2B"),
    "date_iso":       (r"^\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])$", "2024-03-15", "2024-13-01"),
    "date_us":        (r"^(0[1-9]|1[0-2])/(0[1-9]|[12]\d|3[01])/\d{4}$", "03/15/2024", "15/03/2024"),
    "date_dmy":       (r"^(0[1-9]|[12]\d|3[01])-(0[1-9]|1[0-2])-\d{4}$", "15-03-2024", "2024-03-15"),
    "time_24h":       (r"^([01]\d|2[0-3]):[0-5]\d$", "23:59", "24:00"),
    "zip_us":         (r"^\d{5}(-\d{4})?$", "12345-6789", "1234"),
    "pin_india":      (r"^[1-9]\d{5}$", "400001", "012345"),
    "hex_color":      (r"^#(?:[0-9a-fA-F]{3}){1,2}$", "#FF5733", "#GG0000"),
    "uuid":           (r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$", "123e4567-e89b-12d3-a456-426614174000", "123e4567"),
    "username":       (r"^[a-zA-Z][\w]{2,15}$", "john_doe1", "1john"),
    "strong_password":(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[^\w\s]).{8,}$", "Passw0rd!", "password"),
    "credit_card":    (r"^\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}$", "1234-5678-9012-3456", "1234-5678"),
    "ssn":            (r"^\d{3}-\d{2}-\d{4}$", "123-45-6789", "123456789"),
    "currency":       (r"^[$\u20b9\u20ac]\s?\d{1,3}(,\d{3})*(\.\d{2})?$", "$1,234.50", "1234.5$"),
    "hashtag":        (r"^#\w+$", "#python", "# python"),
    "mention":        (r"^@\w{1,15}$", "@guido", "guido"),
    "html_tag":       (r"^<\/?[a-zA-Z][^>]*>$", "<div class='x'>", "<>"),
    "only_digits":    (r"^\d+$", "12345", "12a45"),
}

passed = 0
for name, (pat, good, bad) in PATTERNS.items():
    ok_good = bool(re.match(pat, good))
    ok_bad = not re.match(pat, bad)
    status = "PASS" if (ok_good and ok_bad) else "FAIL"
    passed += status == "PASS"
    print(f"{status}  {name:16} valid={good!r:40} invalid={bad!r}")

print(f"\n{passed}/{len(PATTERNS)} patterns passed")

PASS  email            valid='me@site.com'                            invalid='me@site'
PASS  phone_us         valid='(555) 123-4567'                         invalid='12-34'
PASS  phone_intl       valid='+91 9876543210'                         invalid='9876543210'
PASS  url              valid='https://python.org/doc'                 invalid='python.org'
PASS  ipv4             valid='192.168.0.1'                            invalid='300.1.1.1'
PASS  mac_address      valid='00:1A:2B:3C:4D:5E'                      invalid='00:1A:2B'
PASS  date_iso         valid='2024-03-15'                             invalid='2024-13-01'
PASS  date_us          valid='03/15/2024'                             invalid='15/03/2024'
PASS  date_dmy         valid='15-03-2024'                             invalid='2024-03-15'
PASS  time_24h         valid='23:59'                                  invalid='24:00'
PASS  zip_us           valid='12345-6789'                             invalid='1234'
PASS  pin_india      

In [16]:
# Helper functions built on the library
def is_valid(kind, value):
    return bool(re.match(PATTERNS[kind][0], value))

print(is_valid("email", "test@mail.com"))
print(is_valid("ipv4", "10.0.0.256"))
print(is_valid("strong_password", "Abcdef1!"))

True
False
True


## 5. Apply to Data Cleaning

### 5.1 Clean phone numbers

In [17]:
def clean_phone(p):
    digits = re.sub(r"\D", "", str(p))        # keep digits only
    digits = digits[-10:]                       # drop country code
    if len(digits) != 10:
        return None
    return f"({digits[:3]}) {digits[3:6]}-{digits[6:]}"

for p in ["555-123-4567", "(555) 123 4567", "+1 555.123.4567", "12345"]:
    print(f"{p:18} -> {clean_phone(p)}")

555-123-4567       -> (555) 123-4567
(555) 123 4567     -> (555) 123-4567
+1 555.123.4567    -> (555) 123-4567
12345              -> None


### 5.2 Extract emails

In [18]:
text = "Contact sales@acme.com or support@acme.co.in. Not an email: user@@bad"
print(re.findall(r"[\w.+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+", text))

['sales@acme.com', 'support@acme.co.in']


### 5.3 Parse dates (mixed formats to one standard)

In [19]:
from datetime import datetime

def parse_date(s):
    s = s.strip()
    if m := re.match(r"^(\d{4})-(\d{2})-(\d{2})$", s):                       # 2024-03-15
        y, mo, d = m.groups()
    elif m := re.match(r"^(\d{2})/(\d{2})/(\d{4})$", s):                     # 15/03/2024 (DD/MM/YYYY)
        d, mo, y = m.groups()
    elif m := re.match(r"^(\d{1,2})\s+([A-Za-z]{3})[a-z]*\s+(\d{4})$", s):   # 15 Mar 2024
        d, mon, y = m.groups()
        mo = datetime.strptime(mon, "%b").month
    else:
        return None
    try:
        return datetime(int(y), int(mo), int(d)).strftime("%Y-%m-%d")
    except ValueError:
        return None

for s in ["2024-03-15", "15/03/2024", "5 March 2024", "31/02/2024", "hello"]:
    print(f"{s:14} -> {parse_date(s)}")

2024-03-15     -> 2024-03-15
15/03/2024     -> 2024-03-15
5 March 2024   -> 2024-03-05
31/02/2024     -> None
hello          -> None


### 5.4 Standardize formats and remove special characters

In [20]:
def clean_name(n):
    n = re.sub(r"[^A-Za-z\s'-]", "", n)     # remove special chars
    n = re.sub(r"\s+", " ", n).strip()      # collapse spaces
    return n.title()

def clean_text(t):
    t = t.lower()
    t = re.sub(r"<[^>]+>", "", t)            # strip HTML tags
    t = re.sub(r"http\S+", "", t)           # strip URLs
    t = re.sub(r"[^a-z0-9\s]", "", t)       # strip punctuation / symbols
    return re.sub(r"\s+", " ", t).strip()

print(clean_name("  jOHN   d@oe!!  "))
print(clean_text("<p>Hello,   WORLD!!! Visit https://x.com #python</p>"))

John Doe
hello world visit python


## 6. Practice

### 6.1 Clean customer data

In [21]:
raw = pd.DataFrame({
    "name":  ["  john   SMITH ", "mary_o'neil##", "ALICE   wong", "bob  lee"],
    "email": ["John.Smith@Example.COM", "mary@@mail.com", "alice@site.org ", "bob@lee"],
    "phone": ["555-123-4567", "(555) 987 6543", "+1 555.222.3333", "12345"],
    "joined": ["2024-03-15", "20/04/2023", "5 January 2022", "not a date"],
    "zip":   ["12345", "1234", "98765-4321", "ABCDE"],
})
print("RAW DATA")
raw

RAW DATA


,name,email,phone,joined,zip
0,john SMITH,John.Smith@Example.COM,555-123-4567,2024-03-15,12345
1,mary_o'neil##,mary@@mail.com,(555) 987 6543,20/04/2023,1234
2,ALICE wong,alice@site.org,+1 555.222.3333,5 January 2022,98765-4321
3,bob lee,bob@lee,12345,not a date,ABCDE


In [22]:
clean = raw.copy()
clean["name"]   = clean["name"].apply(lambda n: re.sub(r"\s+", " ", re.sub(r"[^A-Za-z\s'-]", " ", n)).strip().title())
clean["email"]  = clean["email"].str.strip().str.lower()
clean["email_valid"] = clean["email"].apply(lambda e: is_valid("email", e))
clean["phone"]  = clean["phone"].apply(clean_phone)
clean["joined"] = clean["joined"].apply(parse_date)
clean["zip_valid"] = clean["zip"].apply(lambda z: is_valid("zip_us", z))

print("CLEANED DATA")
clean

CLEANED DATA


,name,email,phone,joined,zip,email_valid,zip_valid
0,John Smith,john.smith@example.com,(555) 123-4567,2024-03-15,12345,True,True
1,Mary O'Neil,mary@@mail.com,(555) 987-6543,2023-04-20,1234,False,False
2,Alice Wong,alice@site.org,(555) 222-3333,2022-01-05,98765-4321,True,True
3,Bob Lee,bob@lee,None,None,ABCDE,False,False


### 6.2 Log file parser

In [23]:
log_text = """192.168.1.10 - - [15/Mar/2024:10:15:32 +0000] "GET /index.html HTTP/1.1" 200 1024
192.168.1.11 - - [15/Mar/2024:10:16:01 +0000] "POST /login HTTP/1.1" 401 512
10.0.0.5 - - [15/Mar/2024:10:17:45 +0000] "GET /products?id=5 HTTP/1.1" 200 2048
192.168.1.10 - - [15/Mar/2024:10:18:10 +0000] "GET /missing.png HTTP/1.1" 404 128
10.0.0.7 - - [15/Mar/2024:10:19:22 +0000] "GET /api/data HTTP/1.1" 500 64
192.168.1.11 - - [15/Mar/2024:10:20:05 +0000] "POST /login HTTP/1.1" 200 700
this line is corrupted"""

log_pattern = re.compile(
    r'(?P<ip>\d{1,3}(?:\.\d{1,3}){3}) - - '
    r'\[(?P<time>[^\]]+)\] '
    r'"(?P<method>[A-Z]+) (?P<path>\S+) HTTP/[\d.]+" '
    r'(?P<status>\d{3}) (?P<size>\d+)'
)

rows, bad_lines = [], []
for line in log_text.splitlines():
    m = log_pattern.match(line)
    if m:
        rows.append(m.groupdict())
    else:
        bad_lines.append(line)

logs = pd.DataFrame(rows)
logs["status"] = logs["status"].astype(int)
logs["size"] = logs["size"].astype(int)

print(f"Parsed {len(logs)} lines, skipped {len(bad_lines)} bad line(s)")
logs

Parsed 6 lines, skipped 1 bad line(s)


,ip,time,method,path,status,size
0,192.168.1.10,15/Mar/2024:10:15:32 +0000,GET,/index.html,200,1024
1,192.168.1.11,15/Mar/2024:10:16:01 +0000,POST,/login,401,512
2,10.0.0.5,15/Mar/2024:10:17:45 +0000,GET,/products?id=5,200,2048
3,192.168.1.10,15/Mar/2024:10:18:10 +0000,GET,/missing.png,404,128
4,10.0.0.7,15/Mar/2024:10:19:22 +0000,GET,/api/data,500,64
5,192.168.1.11,15/Mar/2024:10:20:05 +0000,POST,/login,200,700


In [24]:
print("Requests by status:")
print(logs["status"].value_counts().to_string())

print("\nTop IPs:")
print(logs["ip"].value_counts().to_string())

print("\nErrors (4xx / 5xx):")
logs[logs["status"] >= 400][["ip", "path", "status"]]

Requests by status:
status
200    3
401    1
404    1
500    1

Top IPs:
ip
192.168.1.10    2
192.168.1.11    2
10.0.0.5        1
10.0.0.7        1

Errors (4xx / 5xx):


,ip,path,status
1,192.168.1.11,/login,401
3,192.168.1.10,/missing.png,404
4,10.0.0.7,/api/data,500


In [25]:
# Application log example: level, timestamp, message
app_log = """2024-03-15 10:00:01 INFO  User alice logged in
2024-03-15 10:00:05 ERROR Database timeout after 30s
2024-03-15 10:01:12 WARN  Disk usage at 91%
2024-03-15 10:02:40 ERROR Failed to send email to bob@example.com"""

app_pat = re.compile(r"(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) (?P<level>[A-Z]+)\s+(?P<msg>.*)")
app_df = pd.DataFrame([m.groupdict() for m in app_pat.finditer(app_log)])
print(app_df["level"].value_counts().to_string())
print()
print("Emails found in messages:", re.findall(r"[\w.+-]+@[\w.-]+\.\w+", app_log))
app_df

level
ERROR    2
INFO     1
WARN     1

Emails found in messages: ['bob@example.com']


,ts,level,msg
0,2024-03-15 10:00:01,INFO,User alice logged in
1,2024-03-15 10:00:05,ERROR,Database timeout after 30s
2,2024-03-15 10:01:12,WARN,Disk usage at 91%
3,2024-03-15 10:02:40,ERROR,Failed to send email to bob@example.com


## Summary

| Deliverable | Where |
|---|---|
| 20+ regex patterns | Section 4 (22 patterns, auto-tested) |
| Data cleaning scripts | Section 5 and 6.1 |
| Pattern library | `PATTERNS` dict + `is_valid()` |
| Log parser | Section 6.2 (access log + app log) |
| README with regex guide | `README.md` |